[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zshakespear/bd-spectra-onramp/blob/main/notebooks/06_fitting_models.ipynb)

# 6 · Fitting models to spectra

*Brown Dwarf Spectra On-Ramp · about 75 minutes*

**By the end of this notebook you will be able to:**
- fit a grid of models to a spectrum with one free scale factor, and read the best fit and its residuals
- use a χ² map to see which parameters the data pin down and which trade off against each other
- check fitted parameters against what we know about the reference sequence
- tell a model problem from something peculiar about one object, by comparing with the template for its spectral type
- compare indices between a spectrum and its best-fit model, and list what can mimic a chemistry difference
- recognize data problems before interpreting a feature

> **First time here?** Do *File → Save a copy in Drive* before you change anything (notebook 0 explains why).
> Then run the **Setup** cell below. Run cells in order, top to bottom, with **Shift + Enter**.
> **Moving between notebooks:** use the links inside the notebooks (such as **Next** at the bottom). They always open
> the original notebook, so they work from your copy too — then save a copy of the new notebook before you edit it.

In [ ]:
# ▶ SETUP - run this cell first. It is safe to run more than once.
REPO_URL = "https://github.com/zshakespear/bd-spectra-onramp.git"
import os, subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules:                       # running in Google Colab
    ROOT = Path("/content/bd-spectra-onramp")
    if not (ROOT / "bdtools").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only", "--quiet"], check=False)
else:                                                   # running in Jupyter on your own computer
    ROOT = Path.cwd()
    while not (ROOT / "bdtools").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import bdtools as bd
bd.setup_notebook()

## 1. How a model fit works

A brown dwarf you look at usually already has a spectral type, so the question is not *what type is it?* but *how well
do models describe it, and where do they fail?* Here we practise on the references, whose types and distances we know.

**Step 1 — put the model on the data's wavelengths.** A model spectrum is computed at thousands of wavelengths;
SPHEREx measures 102 channels, each covering a small range of wavelengths. The two don't line up, so we **always
adjust the model to fit the data, never the other way round**: for each SPHEREx channel we average the model over the
wavelengths that channel covers.

**Step 2 — scale the model.** A model gives the light leaving the object's surface; we see the object from far away,
so it looks much fainter. We multiply the model by one number, the **scale factor**, chosen to make it match the data
as closely as possible.

**Step 3 — measure how well it matches.** "As closely as possible" means the scale that **minimizes a statistic**.
We use the **reduced χ²** ("chi-squared"): for each channel, take the difference between data and model, divide it by
the uncertainty, square it, and average over all channels. If the model matched perfectly apart from noise, reduced
χ² would be about 1; bigger means a worse match. **G<sub>k</sub>** (Cushing et al. 2008) is a similar statistic that gives wider
channels more weight. For both, **lower is better**. (Following Rustamkulov et al. 2026, we first add a 3% allowance to
every uncertainty for calibration errors.)

**Step 4 — repeat for every model in a grid**, and rank them. Our grid is Sonora Diamondback (Morley et al. 2024): 1440
models covering T<sub>eff</sub>, log g, [M/H] and clouds (f<sub>sed</sub>, or "nc" = cloud-free).

In [ ]:
grid = bd.optional(bd.load_grid, "diamondback")
grid

In [ ]:
# ✏️ SETTINGS YOU CAN CHANGE
DATA_TYPE = "L5"
FIT_WINDOW = None      # None = fit the scale over all channels; e.g. [(1.2, 1.3)] = J peak only
FIXED = {}             # hold parameters fixed, e.g. {"mh": 0.0} or {"clouds": "nc"}

## 2. Fit one reference

**Predict:** this object is an L5. Roughly what T<sub>eff</sub> do you expect the best model to have? (Notebook 4 and
the [spectral sequence page](https://github.com/zshakespear/bd-spectra-onramp/blob/main/field_guide/spectral_sequence.md) give the scale.)

In [ ]:
data = bd.load_reference(DATA_TYPE)
fit = bd.fit_models(data, windows=FIT_WINDOW, **FIXED) if grid is not None else None
if fit is not None:
    print(fit.summary())
    display(fit.table.head(10))

Look at the top ten. Are they all similar models, or do quite different parameter combinations give almost the same
χ²<sub>red</sub>? And notice the size of χ²<sub>red</sub>: even the best model is usually **far** above 1. With a 3% systematic
already included, that means the models do not reproduce the spectrum within its uncertainties. Rustamkulov et al.
(2026) find χ²<sub>red</sub> ≥ 10 for best fits to high-S/N spectra. **Treat best-fit parameters as rough.**

## 3. The best fit and its residuals

In [ ]:
if fit is not None:
    comp = fit.comparison()
    fig, axes = bd.plot_comparison(comp, panel="both", title=f"{data.meta['spectral_type']}: best-fit model")

Read the two lower panels with the band atlas in mind. The pale **coloured shading in every panel marks the absorption
bands** (the key above the plot says which colour is which molecule):

* **ratio** (data / model): where is the data brighter (> 1) or fainter (< 1) than the model, and does each departure sit
  inside a shaded absorber band?
* **significance** ((data − model)/σ): points beyond ±3 are hard to explain by noise alone. Clusters of same-sign points
  inside a band are a *feature*; isolated single points are suspicious (section 9).

### Scaling changes how residuals look
**Predict:** if the same model is scaled to match only at the J peak, will the ratio plot look the same?

In [ ]:
if fit is not None:
    c_j = bd.compare(data, fit.model(), windows=[(1.2, 1.3)])
    fig, axes = bd.plot_comparison(c_j, panel="ratio", title="Same best-fit model, scaled at the J peak only")

Same two spectra, different-looking ratio plot. With J-peak scaling the ratio is 1 at J by construction and every
other difference looks like a slope. **A ratio plot always depends on the scaling**, so say how you scaled whenever
you describe one. Indices (section 7) are ratios within one spectrum and do not depend on it.

## 4. Which parameters do the data pin down? The χ² map

Each cell shows the best χ²<sub>red</sub> among models with that pair of values (the other parameters are free). The star is
the overall best fit. A small dark spot means a well-constrained pair; a long dark valley means the two parameters
**trade off** — a degeneracy (notebook 5).

In [ ]:
# ✏️ SETTINGS YOU CAN CHANGE
MAP_X, MAP_Y = "teff", "logg"     # any two of: "teff", "logg", "mh", "clouds"

In [ ]:
if fit is not None:
    fig, ax = bd.plot_chi2_map(fit, x=MAP_X, y=MAP_Y)
    fig, ax = bd.plot_chi2_map(fit, x="teff", y="clouds")

## 5. Fit the whole sequence

**Predict:** from L0 to Y2, how should the best-fit T<sub>eff</sub> change? What about clouds? Diamondback's coolest model is
900 K. What do you expect for the late-T and Y dwarfs, which are cooler than that?

In [ ]:
seq = bd.load_sequence()
fits = [bd.fit_models(s, windows=FIT_WINDOW, **FIXED) for s in seq] if grid is not None else []
if fits:
    table = bd.fits_table(fits)
    display(table)
    fig, axes = bd.plot_fit_parameters(table)

T<sub>eff</sub> should fall along the sequence — roughly 2300 K at L0 to roughly 1300–1400 K at the end of the L types,
from bolometric luminosities (e.g. Filippazzo et al. 2015). Where does the fit follow that trend, and where does it jump
around? Parameters that jump between neighbouring types (often log g, [M/H], or clouds) are poorly constrained by the
fit, even when the formal best fit names one value. Look also at χ²<sub>red</sub> along the sequence: which types do the models
describe worst?

Three things to look for in the table:

* **The grid edge.** From about T8 onward every fit lands on 900 K, the coolest Diamondback model. A best fit on the
  edge of a grid is **not a measurement**: it only says the object is at least as extreme as the grid allows. (A grid
  built for cooler objects, such as Sonora Flame Skimmer in notebook 5, is the right tool there.)
* **A small χ²<sub>red</sub> is not always a good fit.** The Y dwarfs are faint, with S/N of only a few per channel, so their
  large error bars keep χ²<sub>red</sub> small even when the model is poor. Always look at the residuals too.
* **One huge χ²<sub>red</sub>.** Look at L0. Set `DATA_TYPE = "L0"` in section 1 and look at the significance panel and the quality
  check in section 9: a couple of bad channels with small error bars (near 0.92 and 3.85 µm) dominate χ² and drag
  the scale, so the whole spectrum ends up off. **Check the data before trusting any fit.**

## 6. Is it the object or the models? Compare with the template

A brown dwarf you look at usually already has a spectral type, and each spectral type has a **template**: a reference
spectrum of that type (here, the reference sequence from notebook 4). Fit both the object and its template with the
same grid, and compare their residuals:

* a residual pattern the template **shares** is a **model problem** — the models get every object of that type wrong
  in the same way;
* a residual pattern only your object shows is something about **the object** worth explaining.

Here we practise with the data spectrum and its neighbours in the sequence, which all play the role of the template.

In [ ]:
# ✏️ SETTINGS YOU CAN CHANGE
COMPARE_TYPES = ["L4", "L5", "L6"]

In [ ]:
if fits:
    chosen = [ft for ft in fits if ft.data.meta.get("spectral_type") in COMPARE_TYPES]
    fig, ax = bd.plot_fit_residuals(chosen, title="data / best-fit model for neighbouring types")

Which departures appear in all of them (shared model problems)? Is there anything only one object shows? Remember
that each object has its own best-fit model, so part of the difference between curves is the fit itself — a shared
pattern is the strong clue, not a perfect match.

## 7. Indices: data versus best-fit model

The indices from Rustamkulov et al. (2026, Table 5) are magnitudes,

  I = −2.5 log<sub>10</sub>(F<sub>band</sub> / F<sub>norm</sub>),

with mean F<sub>ν</sub> in a feature region and in the 3.70–4.14 µm normalization window (the nearest thing to a continuum).
Bigger = deeper band. The uncertainty is (2.5/ln 10) √[(σ<sub>F</sub>/F)² + (σ<sub>norm</sub>/F<sub>norm</sub>)²]. A model has no
uncertainty, so a data–model difference is measured against the data's error alone.

In [ ]:
fig, ax = bd.plot_index_windows([data, comp.template] if fit is not None else data)
if fit is not None:
    import pandas as pd
    display(pd.DataFrame([bd.index_difference(data, comp.template, n) for n in ["CH4", "CO2", "CO"]]))

**Before reading an index as chemistry, check its windows on the actual spectra.** The feature regions overlap other
absorbers (see the [band atlas](https://github.com/zshakespear/bd-spectra-onramp/blob/main/field_guide/band_atlas.md)), so one molecule can move another molecule's index. And
the CH<sub>4</sub> index is **not meaningful for Y dwarfs** — CH<sub>4</sub> is undetected there at SPHEREx sensitivity.

### Index planes
An index plane plots **one index against another** for many spectra. The sequence traces a path through the plane.
Where does the "data" spectrum fall relative to the references? Try each pair of indices.

In [ ]:
# ✏️ SETTINGS YOU CAN CHANGE
PLANE_X = "CH4"     # one of: "CH4", "CO2", "CO"
PLANE_Y = "CO"

In [ ]:
fig, ax = bd.plot_index_plane(seq, x=PLANE_X, y=PLANE_Y, highlight=data)

## 8. What can mimic a chemistry difference?

A residual in one band does not have to mean different chemistry. Before you write "more CO<sub>2</sub>", consider:

* **temperature** — changes many bands at once (notebooks 3 and 5);
* **gravity** and **metallicity** — act strongly through H<sub>2</sub> CIA and band depths (notebook 5);
* **clouds** — dim and redden the windows;
* **vertical mixing** — changes CO, CO<sub>2</sub> and CH<sub>4</sub> together (notebook 5);
* **the models themselves** — every grid has its own shortcomings (section 6);
* **unresolved binarity** — two objects of different type blended into one spectrum.

Here is a *simulated* unresolved binary made by adding two references. It is labeled as simulated and is only for practice.

In [ ]:
# ✏️ SETTINGS YOU CAN CHANGE
BINARY_PAIR = ("L2", "L8")      # brighter + fainter component

In [ ]:
p = bd.optional(bd.load_reference, BINARY_PAIR[0]); q = bd.optional(bd.load_reference, BINARY_PAIR[1])
if p is not None and q is not None and grid is not None:
    # put both at 10 pc so their relative brightness is physical, then add them on p's channels
    p10 = p.scaled((p.meta["distance_pc"] / 10) ** 2)
    q10 = q.scaled((q.meta["distance_pc"] / 10) ** 2)
    qf = np.interp(p10.wave_um, q10.wave_um, q10.flux); qe = np.interp(p10.wave_um, q10.wave_um, q10.err)
    blend = p10.copy(flux=p10.flux + qf, err=np.hypot(p10.err, qe),
                     name=f"SIMULATED blend {BINARY_PAIR[0]}+{BINARY_PAIR[1]}")
    bfit = bd.fit_models(blend)
    print(bfit.summary())
    fig, axes = bd.plot_comparison(bfit.comparison(), panel="both",
                                   title="SIMULATED unresolved binary vs its best single model")

A single model can match the blend reasonably well overall but leave structured residuals in specific bands —
exactly the kind of thing that could be misread as a chemistry difference. Compare the best-fit T<sub>eff</sub> with those of the
two components in section 5.

## 9. Data problems: check these *first*

Real spectra contain spurious points even after cleaning (Rustamkulov et al. 2026), and spectra from different
sources are cleaned unevenly. Before interpreting any feature, run a quality check. `bd.quality_report` flags points far from a running
median, gaps, and missing values — it never deletes anything; you decide.

In [ ]:
print(bd.quality_report(data))
flags = bd.running_median_outliers(data)
fig, ax = bd.plot_spectrum(data, log=True, title="Flagged points (red rings) - are they real?")
ax.plot(data.wave_um[flags], data.flux[flags], "o", mfc="none", mec="red", ms=10, mew=2);

Now some **simulated** problems, so you know what each looks like against the best-fit model. Each is applied to a
copy of the data and labeled as simulated.

In [ ]:
rng = np.random.default_rng(1)
problems = {}
# 1. isolated outlier points
f = data.flux.copy(); idx = rng.choice(len(f), 3, replace=False); f[idx] *= rng.choice([0.6, 1.5], 3)
problems["isolated outliers"] = data.copy(flux=f)
# 2. a gap (channels missing)
keep = ~((data.wave_um > 2.0) & (data.wave_um < 2.6))
problems["gap at 2.0-2.6 um"] = data.select(keep)
# 3. uneven sampling (random channels missing)
problems["uneven sampling"] = data.select(rng.random(len(data)) > 0.3)
# 4. contamination by a bluer neighbour (an L0 adding ~20% of the data's J-band flux)
nb = bd.optional(bd.load_reference, "L0")
if nb is not None:
    nbf = np.interp(data.wave_um, nb.wave_um, nb.flux)
    j = (data.wave_um > 1.2) & (data.wave_um < 1.3)
    problems["contaminated by a neighbour"] = data.copy(flux=data.flux + 0.2 * nbf * data.flux[j].mean() / nbf[j].mean())
for label, bad in problems.items():
    if fit is not None:
        c = bd.compare(bad.copy(name=f"SIMULATED: {label}"), fit.model())
        bd.plot_comparison(c, panel="significance", title=f"SIMULATED problem: {label}")

What to remember:

| Problem | What it looks like |
|---|---|
| isolated outlier | one point far off, neighbours fine; big single spike in significance |
| gap | missing stretch; an index or fit can silently use fewer channels |
| uneven sampling | irregular spacing; fine for fitting, but check every region still has points |
| contamination | a smooth, broad excess (often at short wavelengths for a bluer neighbour); not confined to one band |

See the [field guide page on data problems](https://github.com/zshakespear/bd-spectra-onramp/blob/main/field_guide/data_problems.md).

## 10. Where this leads

You now have the skills to examine a real brown-dwarf spectrum: reading a spectrum, the band atlas, the sequence,
the effect of each parameter, model fits with uncertainties, comparing with the template for its type, indices, and
data-quality checks. The next step, which your instructor will give you, applies them to real SPHEREx spectra and asks
you to explain, in words, where and why an object departs from its best-fit model.

### Exercise 6.1
Set `FIXED = {"clouds": "nc"}` (cloud-free models only) and re-run sections 2–5. How do the best-fit T<sub>eff</sub> and
χ²<sub>red</sub> change, and for which types most?

<details><summary><b>Answer</b> (try first, then click)</summary>

Without clouds the models cannot dim and redden the windows, so the fit usually compensates with a different
T<sub>eff</sub> (often cooler for mid/late L dwarfs) and χ²<sub>red</sub> gets worse, most for the dusty mid/late L types.
A parameter you hold fixed pushes the error into the others — another reason to treat best-fit values as rough.

</details>

### Exercise 6.2
An object's CO<sub>2</sub> index is 0.15 ± 0.04 mag larger than its best-fit model's, and the template for its type shows no
such difference. List three explanations other than "more CO<sub>2</sub>".

<details><summary><b>Answer</b> (try first, then click)</summary>

Any three of: a different temperature, gravity, or metallicity than the fit found (the fit balances all channels, not
this band); vertical mixing changing CO/CO<sub>2</sub>; clouds; an overlapping absorber in the index window; an unresolved
companion; a bad point or gap in the 4.18–4.35 µm region or the normalization window. (And the difference is ~4σ —
check that it is not one bad channel.) Because the template does *not* share it, a pure model problem is less likely.

</details>

## Summary

- Fit a grid: bin each model onto the data's channels, one free scale, 3% systematic, rank by χ²<sub>red</sub>.
- Best fits typically still have χ²<sub>red</sub> ≫ 1; best-fit parameters are rough, and χ² maps show degeneracies.
- Ratio plots depend on the scaling; indices do not.
- Residuals shared with the template are model problems; residuals only your object shows are worth explaining.
- Temperature, gravity, metallicity, clouds, mixing, the models, and binarity can all mimic a chemistry difference.
- Check for outliers, gaps, uneven sampling, and contamination before interpreting any feature.
- Field guide: [Indices](https://github.com/zshakespear/bd-spectra-onramp/blob/main/field_guide/indices.md), [Data problems](https://github.com/zshakespear/bd-spectra-onramp/blob/main/field_guide/data_problems.md)

**Field guide:** the pages linked above are the short versions to keep open while you work.

## Self-check

Answer these without scrolling up. If one is hard, revisit that section.

1. Why is each model averaged onto the data's channels before fitting?
<details><summary><b>Answer</b> (try first, then click)</summary>

SPHEREx blurs narrow model features together; comparing an unbinned model with SPHEREx data is not fair.

</details>

2. What does a long dark valley in a χ² map mean?
<details><summary><b>Answer</b> (try first, then click)</summary>

The two parameters trade off (a degeneracy); the data do not pin down either alone.

</details>

3. A residual appears in an object and in its template. What is the likely cause?
<details><summary><b>Answer</b> (try first, then click)</summary>

The models — they get every object of that type wrong in the same way.

</details>

4. When is the CH<sub>4</sub> index not meaningful?
<details><summary><b>Answer</b> (try first, then click)</summary>

For Y dwarfs (CH<sub>4</sub> undetected at SPHEREx sensitivity).

</details>

5. What does an isolated outlier look like in a significance plot?
<details><summary><b>Answer</b> (try first, then click)</summary>

A single large spike with normal neighbours, often outside any band.

</details>

**Next:** ask your instructor for the next step.